# Assignment 2C — Part A: Four Base Prompt Patterns
Implements Zero-shot, Few-shot, RAG, Self-Critique, and the ten-query benchmark. Each code cell starts with a comment describing its purpose. The reference dataset uses instruction/response fields and is streamed for the first ten questions.


## Clone the GitHub repository

In [1]:
from pathlib import Path
import subprocess
import os

REPO_URL = "https://github.com/tusharchouhan/banking-compliance-llm-assignment-1a.git"
PROJECT = Path("/content/banking-compliance-llm-assignment-1a")

if not (PROJECT / "src").exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)

os.chdir(PROJECT)
print("Project:", Path.cwd())
print("src exists:", (PROJECT / "src").exists())


Project: /content/banking-compliance-llm-assignment-1a
src exists: True


In [2]:
!ls

'Assignment Documentation'		     notebooks	        results
 Banking-compliance-llm-assignment-1a.docx   README.md	        run_pipeline.py
 data					     reports	        src
 models					     requirements.txt


In [5]:
!python -m pip install -r requirements.txt
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 10.0 MB/s  0:00:02


In [ ]:
# This cell installs the model inference, PEFT, embedding, and FAISS retrieval dependencies.%pip install -q transformers peft accelerate sentence-transformers faiss-cpu

In [6]:
# This cell loads the Assignment 1 model or a configurable Hugging Face fallback, reads the first ten benchmark examples, and builds a FAISS index over cleaned domain text.
from pathlib import Path
import os, json, time, re, warnings
import torch, numpy as np, pandas as pd, faiss
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sentence_transformers import SentenceTransformer

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = Path("/content/LLM_Project_1_2")
DATA = ROOT / "data"
MODEL_ID = os.getenv("MODEL_ID", "HuggingFaceTB/SmolLM2-360M")
BASE_PATH = os.getenv("BASE_MODEL_DIR", str(ROOT / "models" / "base_model"))
ADAPTER_PATH = os.getenv("ADAPTER_DIR", str(ROOT / "models" / "adapter_A"))
if not Path(BASE_PATH, "config.json").exists():
    BASE_PATH = MODEL_ID
tokenizer = AutoTokenizer.from_pretrained(BASE_PATH)
model = AutoModelForCausalLM.from_pretrained(
    BASE_PATH,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)
adapter_loaded = Path(ADAPTER_PATH, "adapter_config.json").exists()
if adapter_loaded:
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
else:
    warnings.warn("Assignment 1 adapter not found; using base model. Set ADAPTER_DIR to your uploaded adapter for required final results.")
model.eval()

def generate(prompt, max_new_tokens=256):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    started = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    answer = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return answer, int(output.shape[-1]), time.perf_counter() - started

dataset_path = DATA / "instruction_dataset" / "instruction_dataset.jsonl"
if not dataset_path.exists():
    raise FileNotFoundError(f"Expected benchmark file: {dataset_path}")
benchmark = []
with dataset_path.open(encoding="utf-8") as stream:
    for line in stream:
        if line.strip():
            row = json.loads(line)
            benchmark.append({
                "query": row.get("query", row.get("instruction", row.get("question", ""))),
                "answer": row.get("answer", row.get("response", ""))
            })
        if len(benchmark) == 10:
            break
if len(benchmark) < 10:
    raise ValueError("The benchmark file must contain at least ten question-answer records.")

def read_chunks(folder=DATA / "cleaned_txt", chunk_words=220, overlap=40):
    chunks = []
    step = chunk_words - overlap
    for file in folder.rglob("*.txt"):
        words = file.read_text(encoding="utf-8", errors="ignore").split()
        for start in range(0, len(words), step):
            text = " ".join(words[start:start + chunk_words])
            if text:
                chunks.append({"source": str(file.relative_to(folder)), "text": text})
    if not chunks:
        raise FileNotFoundError(f"No cleaned .txt corpus found under {folder}")
    return chunks

chunks = read_chunks()
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
vectors = embedder.encode([item["text"] for item in chunks], normalize_embeddings=True, show_progress_bar=True).astype("float32")
faiss_index = faiss.IndexFlatIP(vectors.shape[1])
faiss_index.add(vectors)

def retrieve(query, k=2):
    query_vector = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = faiss_index.search(query_vector, min(k, len(chunks)))
    return [{"score": float(score), **chunks[idx]} for score, idx in zip(scores[0], ids[0]) if idx >= 0]

print(f"Questions: {len(benchmark)} | text chunks: {len(chunks)} | GPU: {torch.cuda.is_available()} | adapter loaded: {adapter_loaded}")



config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

... (17 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/314 [00:00<?, ?it/s]

Questions: 10 | text chunks: 10044 | GPU: False | adapter loaded: False


In [7]:
# This cell implements the four required base patterns with comparable answer, token, and latency outputs. RAG also returns its retrieved chunks for inspection.
def zero_shot(query):
    return generate(query)

def few_shot(query, examples=None, k=3):
    examples = (examples or benchmark)[:k]
    demonstrations = "\n\n".join(
        f"Question: {item['query']}\nAnswer: {item['answer']}" for item in examples
    )
    prompt = f"Use these worked examples as guidance.\n{demonstrations}\n\nQuestion: {query}\nAnswer:"
    return generate(prompt)

def rag(query):
    hits = retrieve(query, k=2)
    context = "\n\n".join(f"[{item['source']}] {item['text']}" for item in hits)
    prompt = f"Answer using only the evidence below. If it is insufficient, say so.\nEvidence:\n{context}\n\nQuestion: {query}\nAnswer:"
    result = generate(prompt)
    return result, hits

def self_critique(query):
    draft, tokens1, latency1 = generate(f"Answer the question accurately: {query}")
    critique, tokens2, latency2 = generate(
        f"Check the draft for errors, missing points, and unsupported claims.\nQuestion: {query}\nDraft: {draft}\nCritique:"
    )
    revised, tokens3, latency3 = generate(
        f"Revise the draft using the critique. Do not add unsupported claims.\nQuestion: {query}\nDraft: {draft}\nCritique: {critique}\nRevised answer:"
    )
    return revised, tokens1 + tokens2 + tokens3, latency1 + latency2 + latency3, draft, critique

def accuracy_score(answer, reference):
    # This overlap estimate is an aid only; manually review and assign the official 0–2 rubric.
    answer_words = set(re.findall(r"\w+", answer.lower()))
    reference_words = set(re.findall(r"\w+", reference.lower()))
    overlap = len(answer_words & reference_words) / max(1, len(reference_words))
    return 2 if overlap >= 0.65 else (1 if overlap >= 0.30 else 0)



In [8]:
# This cell displays the retrieved top-two chunks for five queries and the draft/critique/revision for two questions, as requested by the marking checklist.
for item in benchmark[:5]:
    (rag_answer, rag_tokens, rag_latency), retrieved = rag(item["query"])
    print("QUERY:", item["query"])
    print("TOP TWO RETRIEVED CHUNKS:")
    for chunk in retrieved:
        print("SOURCE:", chunk["source"], "SCORE:", round(chunk["score"], 4))
        print(chunk["text"][:900], "\\n")
for item in benchmark[:2]:
    revised, tokens, latency, draft, critique = self_critique(item["query"])
    print("QUERY:", item["query"])
    print("DRAFT:", draft, "\\nCRITIQUE:", critique, "\\nREVISED:", revised)

QUERY: What should a compliance analyst know about vi who neither himself or herself nor whose relative from this provision?
TOP TWO RETRIEVED CHUNKS:
SOURCE: SEBI/1776685013526.txt SCORE: 0.5332
(5) The Compliance Officer, Chief Executive Officer and the Chief Financial Officer shall provide the compliance certificate to the Board of Directors of the Manager as specified in in Part B of Schedule VIII along with supporting evidence thereof. (6) The Board of Directors of the Manager shall set forth clearly the recommendation of the Manager in the notice to the unitholders fo r each item referred to in sub regulation (6) of regulation 22 of these regulations. Vigil Mechanism 26C. (1) The Manager shall formulate a vigil mechanism, including a whistle blower policy for directors and employees to report genuine concerns. (2) The vigil mechanism shall provide for adequate safeguards against victimization of Director(s) or employee(s) or any other person who avail the mechanism and also provi

In [9]:
# This cell runs all four methods against the same ten benchmark questions and produces the required accuracy, token, and latency table.
records = []
for item in benchmark:
    query, reference = item["query"], item["answer"]
    method_outputs = {
        "Zero-shot": zero_shot(query),
        "Few-shot": few_shot(query),
        "RAG": rag(query)[0],
        "Self-Critique": self_critique(query)[:3]
    }
    for method, (answer, tokens, latency) in method_outputs.items():
        records.append({
            "query": query, "pattern": method, "answer": answer,
            "tokens": tokens, "latency_sec": latency,
            "preliminary_accuracy_0_2": accuracy_score(answer, reference),
            "human_accuracy_0_2": None
        })
part_a_results = pd.DataFrame(records)
display(part_a_results)
part_a_summary = part_a_results.groupby("pattern").agg(
    avg_preliminary_accuracy=("preliminary_accuracy_0_2", "mean"),
    total_tokens=("tokens", "sum"),
    avg_latency_sec=("latency_sec", "mean")
).reset_index()
display(part_a_summary)



,query,pattern,answer,tokens,latency_sec,preliminary_accuracy_0_2,human_accuracy_0_2
0,What should a compliance analyst know about vi...,Zero-shot,The compliance analyst should know that the co...,276,55.558738,0,None
1,What should a compliance analyst know about vi...,Few-shot,"(vi) who, neither himself or herself, nor whos...",1132,79.850475,1,None
2,What should a compliance analyst know about vi...,RAG,The compliance analyst should know about the f...,891,70.397653,0,None
3,What should a compliance analyst know about vi...,Self-Critique,The\nRevised answer:\n\nThe\nRevised answer:\n...,1650,201.489723,0,None
4,Summarize the compliance requirement described...,Zero-shot,The following is a summary of the compliance r...,277,60.551349,2,None
5,Summarize the compliance requirement described...,Few-shot,"Prior to its omission, it read as “Investor”.\...",1132,78.730203,2,None
6,Summarize the compliance requirement described...,RAG,The SEBI (Listing Obligations and Disclosure R...,1128,84.886578,2,None
7,Summarize the compliance requirement described...,Self-Critique,The answer should be 100 words or less.\n\nThe...,1653,201.579196,0,None
8,What should a compliance analyst know about Th...,Zero-shot,a. The number of employees who are affected by...,276,60.203902,0,None
9,What should a compliance analyst know about Th...,Few-shot,The following details regarding the issue to b...,1132,84.488246,2,None


,pattern,avg_preliminary_accuracy,total_tokens,avg_latency_sec
0,Few-shot,1.1,11310,83.021101
1,RAG,1.2,9689,80.166760
2,Self-Critique,0.3,16493,206.052683
3,Zero-shot,0.6,2753,60.477018


## Required 75-word analysis
After human review, explain which pattern has the best accuracy per token and whether Self-Critique's three-call cost is justified. Fill in the human_accuracy_0_2 column; automated overlap scores are preliminary only.


### Save and Push Changes Back to Git
Configure your Git credentials below, stage all changes, commit, and push back to your remote repository.

In [10]:
import subprocess
from google.colab import userdata

# Configure your Git credentials
USER_NAME = "tusharchouhan"
USER_EMAIL = "tushar.chouhan@hotmail.com"

try:
    # Retrieve Git token securely using the secret key "NLP"
    git_token = userdata.get('NLP')

    # Set global user config
    subprocess.run(["git", "config", "--global", "user.name", USER_NAME], check=True)
    subprocess.run(["git", "config", "--global", "user.email", USER_EMAIL], check=True)

    # Re-configure remote URL securely with the retrieved token
    auth_url = f"https://{git_token}@github.com/tusharchouhan/banking-compliance-llm-assignment-1a.git"
    subprocess.run(["git", "remote", "set-url", "origin", auth_url], check=True)

    # Stage all modified and untracked files/folders
    subprocess.run(["git", "add", "-A"], check=True)

    # Commit the changes
    commit_message = "Save generated notebook updates, results, and evaluation artifacts"
    result = subprocess.run(["git", "commit", "-m", commit_message], capture_output=True, text=True)
    print("Commit Output:", result.stdout)

    # Push changes back to the origin
    push_result = subprocess.run(["git", "push", "origin", "main"], capture_output=True, text=True)
    print("Push Output:", push_result.stdout)
    if push_result.stderr:
        print("Push Errors/Warnings:", push_result.stderr)

except Exception as e:
    print(f"An error occurred during git push: {e}")

Commit Output: On branch master
Your branch is up to date with 'origin/master'.

nothing to commit, working tree clean

Push Output: 
Push Errors/Warnings: error: src refspec main does not match any
error: failed to push some refs to 'https://github.com/tusharchouhan/banking-compliance-llm-assignment-1a.git'

